# Step 1 : Construction de la Knowledge Base Privée

Ce notebook construit notre base de connaissances sur la musique.
Un triplet c'est toujours : (Sujet, Prédicat, Objet)

### Installation

In [ ]:
!pip install rdflib requests

### Imports

In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD, FOAF
import json
import re
from urllib.parse import quote

### Namespaces

In [ ]:
EX  = Namespace("http://musickg.example.org/resource/")
EXO = Namespace("http://musickg.example.org/ontology/")
DBO = Namespace("http://dbpedia.org/ontology/")
DBR = Namespace("http://dbpedia.org/resource/")
WD  = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

print("Exemple d'URI generee :")
print(EX["The_Beatles"])
print(EXO["hasGenre"])

### Données des 45 artistes

J'ai choisi un mélange de 10 artistes francophones et 35 artistes anglophones.
Le critère principal est la richesse de leur documentation sur Wikidata,
car la qualité de l'expansion SPARQL au Step 4 en dépend directement.
Les artistes francophones retenus (Daft Punk, Stromae, Céline Dion, Edith Piaf...)
sont parmi les mieux documentés de la scène francophone sur Wikidata.

In [ ]:
ARTISTS_DATA = [

    # ===== ARTISTES FRANCOPHONES =====

    {
        "name": "Daft Punk",
        "type": "Group",
        "country": "FR",
        "begin": "1993",
        "end": "2021",
        "genres": ["electronic", "house", "french house"],
        "label": "Virgin Records",
        "members": ["Thomas Bangalter", "Guy-Manuel de Homem-Christo"],
        "wikidata": "Q184803",
        "albums": [("Homework", "1997"), ("Discovery", "2001"), ("Random Access Memories", "2013")]
    },
    {
        "name": "Stromae",
        "type": "Person",
        "country": "BE",
        "begin": "2009",
        "end": None,
        "genres": ["electro", "pop", "hip hop"],
        "label": "Mosaert",
        "members": [],
        "wikidata": "Q193337",
        "albums": [("Cheese", "2010"), ("Racine Carree", "2013"), ("Multitude", "2022")]
    },
    {
        "name": "Celine Dion",
        "type": "Person",
        "country": "CA",
        "begin": "1980",
        "end": None,
        "genres": ["pop", "chanson", "R&B"],
        "label": "Sony Music",
        "members": [],
        "wikidata": "Q3083",
        "albums": [("Falling into You", "1996"), ("Lets Talk About Love", "1997"), ("A New Day Has Come", "2002")]
    },
    {
        "name": "Edith Piaf",
        "type": "Person",
        "country": "FR",
        "begin": "1935",
        "end": "1963",
        "genres": ["chanson", "pop"],
        "label": "Columbia Records",
        "members": [],
        "wikidata": "Q1631",
        "albums": [("La Vie en Rose", "1947"), ("Non je ne regrette rien", "1960"), ("Hymne a lamour", "1950")]
    },
    {
        "name": "Serge Gainsbourg",
        "type": "Person",
        "country": "FR",
        "begin": "1954",
        "end": "1991",
        "genres": ["chanson", "pop", "reggae"],
        "label": "Philips Records",
        "members": [],
        "wikidata": "Q232969",
        "albums": [("Histoire de Melody Nelson", "1971"), ("L Homme a tete de chou", "1976"), ("Aux armes et caetera", "1979")]
    },
    {
        "name": "Charles Aznavour",
        "type": "Person",
        "country": "FR",
        "begin": "1946",
        "end": "2018",
        "genres": ["chanson", "pop"],
        "label": "Barclay Records",
        "members": [],
        "wikidata": "Q170749",
        "albums": [("Je men remets a toi", "1963"), ("La Boheme", "1965"), ("She", "1974")]
    },
    {
        "name": "Jacques Brel",
        "type": "Person",
        "country": "BE",
        "begin": "1953",
        "end": "1978",
        "genres": ["chanson", "pop"],
        "label": "Philips Records",
        "members": [],
        "wikidata": "Q192669",
        "albums": [("Ne me quitte pas", "1959"), ("Amsterdam", "1964"), ("Ces gens-la", "1966")]
    },
    {
        "name": "MC Solaar",
        "type": "Person",
        "country": "FR",
        "begin": "1990",
        "end": None,
        "genres": ["rap", "hip hop", "jazz rap"],
        "label": "Polydor",
        "members": [],
        "wikidata": "Q520922",
        "albums": [("Qui seme le vent recolte le tempo", "1991"), ("Prose Combat", "1994"), ("Chapitre 7", "2007")]
    },
    {
        "name": "Air",
        "type": "Group",
        "country": "FR",
        "begin": "1995",
        "end": None,
        "genres": ["electronic", "ambient", "dream pop"],
        "label": "Source Records",
        "members": ["Nicolas Godin", "Jean-Benoit Dunckel"],
        "wikidata": "Q389488",
        "albums": [("Moon Safari", "1998"), ("The Virgin Suicides", "2000"), ("Talkie Walkie", "2004")]
    },
    {
        "name": "Mylene Farmer",
        "type": "Person",
        "country": "FR",
        "begin": "1984",
        "end": None,
        "genres": ["pop", "synth-pop", "electronic"],
        "label": "Polydor",
        "members": [],
        "wikidata": "Q235196",
        "albums": [("Cendres de lune", "1986"), ("Ainsi soit je", "1988"), ("Anamorphosee", "1995")]
    },

    # ===== ARTISTES ANGLOPHONES =====

    {
        "name": "The Beatles",
        "type": "Group",
        "country": "GB",
        "begin": "1960",
        "end": "1970",
        "genres": ["rock", "pop", "psychedelic rock"],
        "label": "Parlophone",
        "members": ["John Lennon", "Paul McCartney", "George Harrison", "Ringo Starr"],
        "wikidata": "Q1299",
        "albums": [("Abbey Road", "1969"), ("Let It Be", "1970"), ("Revolver", "1966")]
    },
    {
        "name": "Pink Floyd",
        "type": "Group",
        "country": "GB",
        "begin": "1965",
        "end": "2014",
        "genres": ["progressive rock", "psychedelic rock", "art rock"],
        "label": "EMI",
        "members": ["Syd Barrett", "Roger Waters", "David Gilmour", "Nick Mason", "Richard Wright"],
        "wikidata": "Q2306",
        "albums": [("The Dark Side of the Moon", "1973"), ("Wish You Were Here", "1975"), ("The Wall", "1979")]
    },
    {
        "name": "Led Zeppelin",
        "type": "Group",
        "country": "GB",
        "begin": "1968",
        "end": "1980",
        "genres": ["hard rock", "heavy metal", "blues rock"],
        "label": "Atlantic Records",
        "members": ["Jimmy Page", "Robert Plant", "John Paul Jones", "John Bonham"],
        "wikidata": "Q2331",
        "albums": [("Led Zeppelin IV", "1971"), ("Physical Graffiti", "1975"), ("Houses of the Holy", "1973")]
    },
    {
        "name": "Radiohead",
        "type": "Group",
        "country": "GB",
        "begin": "1985",
        "end": None,
        "genres": ["alternative rock", "art rock", "electronic"],
        "label": "Parlophone",
        "members": ["Thom Yorke", "Jonny Greenwood", "Colin Greenwood", "Ed O Brien", "Philip Selway"],
        "wikidata": "Q128309",
        "albums": [("OK Computer", "1997"), ("Kid A", "2000"), ("In Rainbows", "2007")]
    },
    {
        "name": "David Bowie",
        "type": "Person",
        "country": "GB",
        "begin": "1962",
        "end": "2016",
        "genres": ["glam rock", "art rock", "pop"],
        "label": "RCA Records",
        "members": [],
        "wikidata": "Q5383",
        "albums": [("Ziggy Stardust", "1972"), ("Heroes", "1977"), ("Lets Dance", "1983")]
    },
    {
        "name": "Michael Jackson",
        "type": "Person",
        "country": "US",
        "begin": "1964",
        "end": "2009",
        "genres": ["pop", "soul", "funk", "R&B"],
        "label": "Epic Records",
        "members": [],
        "wikidata": "Q2831",
        "albums": [("Thriller", "1982"), ("Bad", "1987"), ("Dangerous", "1991")]
    },
    {
        "name": "Bob Dylan",
        "type": "Person",
        "country": "US",
        "begin": "1961",
        "end": None,
        "genres": ["folk", "rock", "blues", "country"],
        "label": "Columbia Records",
        "members": [],
        "wikidata": "Q392",
        "albums": [("The Freewheelin Bob Dylan", "1963"), ("Highway 61 Revisited", "1965"), ("Blood on the Tracks", "1975")]
    },
    {
        "name": "Rolling Stones",
        "type": "Group",
        "country": "GB",
        "begin": "1962",
        "end": None,
        "genres": ["rock", "blues rock", "hard rock"],
        "label": "Decca Records",
        "members": ["Mick Jagger", "Keith Richards", "Charlie Watts", "Ronnie Wood"],
        "wikidata": "Q11036",
        "albums": [("Let It Bleed", "1969"), ("Sticky Fingers", "1971"), ("Exile on Main St", "1972")]
    },
    {
        "name": "Queen",
        "type": "Group",
        "country": "GB",
        "begin": "1970",
        "end": None,
        "genres": ["rock", "glam rock", "hard rock", "progressive rock"],
        "label": "EMI",
        "members": ["Freddie Mercury", "Brian May", "Roger Taylor", "John Deacon"],
        "wikidata": "Q15862",
        "albums": [("A Night at the Opera", "1975"), ("News of the World", "1977"), ("Jazz", "1978")]
    },
    {
        "name": "Nirvana",
        "type": "Group",
        "country": "US",
        "begin": "1987",
        "end": "1994",
        "genres": ["grunge", "alternative rock", "punk rock"],
        "label": "DGC Records",
        "members": ["Kurt Cobain", "Krist Novoselic", "Dave Grohl"],
        "wikidata": "Q11649",
        "albums": [("Bleach", "1989"), ("Nevermind", "1991"), ("In Utero", "1993")]
    },
    {
        "name": "Beyonce",
        "type": "Person",
        "country": "US",
        "begin": "1997",
        "end": None,
        "genres": ["pop", "R&B", "soul"],
        "label": "Columbia Records",
        "members": [],
        "wikidata": "Q83405",
        "albums": [("Dangerously in Love", "2003"), ("Lemonade", "2016"), ("Renaissance", "2022")]
    },
    {
        "name": "Eminem",
        "type": "Person",
        "country": "US",
        "begin": "1992",
        "end": None,
        "genres": ["hip hop", "rap", "hardcore hip hop"],
        "label": "Interscope Records",
        "members": [],
        "wikidata": "Q1199",
        "albums": [("The Slim Shady LP", "1999"), ("The Marshall Mathers LP", "2000"), ("The Eminem Show", "2002")]
    },
    {
        "name": "Jay-Z",
        "type": "Person",
        "country": "US",
        "begin": "1989",
        "end": None,
        "genres": ["hip hop", "rap", "east coast hip hop"],
        "label": "Roc-A-Fella Records",
        "members": [],
        "wikidata": "Q170930",
        "albums": [("Reasonable Doubt", "1996"), ("The Blueprint", "2001"), ("The Black Album", "2003")]
    },
    {
        "name": "Kanye West",
        "type": "Person",
        "country": "US",
        "begin": "1996",
        "end": None,
        "genres": ["hip hop", "rap", "gospel rap"],
        "label": "Roc-A-Fella Records",
        "members": [],
        "wikidata": "Q44088",
        "albums": [("The College Dropout", "2004"), ("My Beautiful Dark Twisted Fantasy", "2010"), ("Yeezus", "2013")]
    },
    {
        "name": "Adele",
        "type": "Person",
        "country": "GB",
        "begin": "2006",
        "end": None,
        "genres": ["pop", "soul", "R&B"],
        "label": "XL Recordings",
        "members": [],
        "wikidata": "Q81771",
        "albums": [("19", "2008"), ("21", "2011"), ("30", "2021")]
    },
    {
        "name": "Coldplay",
        "type": "Group",
        "country": "GB",
        "begin": "1996",
        "end": None,
        "genres": ["alternative rock", "pop rock", "post-Britpop"],
        "label": "Parlophone",
        "members": ["Chris Martin", "Jonny Buckland", "Guy Berryman", "Will Champion"],
        "wikidata": "Q45188",
        "albums": [("Parachutes", "2000"), ("A Rush of Blood to the Head", "2002"), ("XY", "2005")]
    },
    {
        "name": "Arctic Monkeys",
        "type": "Group",
        "country": "GB",
        "begin": "2002",
        "end": None,
        "genres": ["indie rock", "post-punk revival", "alternative rock"],
        "label": "Domino Records",
        "members": ["Alex Turner", "Matt Helders", "Nick O Malley", "Jamie Cook"],
        "wikidata": "Q134541",
        "albums": [("Whatever People Say I Am", "2006"), ("AM", "2013"), ("Tranquility Base Hotel", "2018")]
    },
    {
        "name": "Tame Impala",
        "type": "Group",
        "country": "AU",
        "begin": "2007",
        "end": None,
        "genres": ["psychedelic rock", "neo-psychedelia", "synth-pop"],
        "label": "Modular Recordings",
        "members": ["Kevin Parker"],
        "wikidata": "Q869612",
        "albums": [("Innerspeaker", "2010"), ("Lonerism", "2012"), ("Currents", "2015")]
    },
    {
        "name": "Amy Winehouse",
        "type": "Person",
        "country": "GB",
        "begin": "2003",
        "end": "2011",
        "genres": ["soul", "jazz", "R&B"],
        "label": "Island Records",
        "members": [],
        "wikidata": "Q131272",
        "albums": [("Frank", "2003"), ("Back to Black", "2006")]
    },
    {
        "name": "Miles Davis",
        "type": "Person",
        "country": "US",
        "begin": "1944",
        "end": "1991",
        "genres": ["jazz", "bebop", "modal jazz", "jazz fusion"],
        "label": "Columbia Records",
        "members": [],
        "wikidata": "Q93341",
        "albums": [("Kind of Blue", "1959"), ("Bitches Brew", "1970"), ("Sketches of Spain", "1960")]
    },
    {
        "name": "Nina Simone",
        "type": "Person",
        "country": "US",
        "begin": "1954",
        "end": "2003",
        "genres": ["jazz", "soul", "blues", "classical"],
        "label": "Colpix Records",
        "members": [],
        "wikidata": "Q128439",
        "albums": [("Little Girl Blue", "1958"), ("I Put a Spell on You", "1965"), ("Pastel Blues", "1965")]
    },
    {
        "name": "Elvis Presley",
        "type": "Person",
        "country": "US",
        "begin": "1954",
        "end": "1977",
        "genres": ["rock and roll", "rockabilly", "pop", "country"],
        "label": "RCA Records",
        "members": [],
        "wikidata": "Q303",
        "albums": [("Elvis Presley", "1956"), ("From Elvis in Memphis", "1969"), ("Elvis Is Back", "1960")]
    },
    {
        "name": "Bob Marley",
        "type": "Person",
        "country": "JM",
        "begin": "1963",
        "end": "1981",
        "genres": ["reggae", "ska", "rocksteady"],
        "label": "Island Records",
        "members": [],
        "wikidata": "Q76",
        "albums": [("Catch a Fire", "1973"), ("Exodus", "1977"), ("Uprising", "1980")]
    },
    {
        "name": "Aretha Franklin",
        "type": "Person",
        "country": "US",
        "begin": "1956",
        "end": "2018",
        "genres": ["soul", "gospel", "R&B", "jazz"],
        "label": "Atlantic Records",
        "members": [],
        "wikidata": "Q5990",
        "albums": [("I Never Loved a Man the Way I Love You", "1967"), ("Lady Soul", "1968"), ("Amazing Grace", "1972")]
    },
    {
        "name": "Frank Sinatra",
        "type": "Person",
        "country": "US",
        "begin": "1937",
        "end": "1998",
        "genres": ["jazz", "pop", "swing", "big band"],
        "label": "Capitol Records",
        "members": [],
        "wikidata": "Q40912",
        "albums": [("In the Wee Small Hours", "1955"), ("Songs for Swingin Lovers", "1956"), ("Come Fly with Me", "1958")]
    },
    {
        "name": "Kendrick Lamar",
        "type": "Person",
        "country": "US",
        "begin": "2003",
        "end": None,
        "genres": ["hip hop", "conscious hip hop"],
        "label": "Top Dawg Entertainment",
        "members": [],
        "wikidata": "Q205303",
        "albums": [("good kid mAAd city", "2012"), ("To Pimp a Butterfly", "2015"), ("DAMN", "2017")]
    },
    {
        "name": "Drake",
        "type": "Person",
        "country": "CA",
        "begin": "2001",
        "end": None,
        "genres": ["hip hop", "R&B", "trap"],
        "label": "Young Money",
        "members": [],
        "wikidata": "Q33777",
        "albums": [("Take Care", "2011"), ("Nothing Was the Same", "2013"), ("Scorpion", "2018")]
    },
    {
        "name": "Rihanna",
        "type": "Person",
        "country": "BB",
        "begin": "2003",
        "end": None,
        "genres": ["pop", "R&B", "dancehall"],
        "label": "Def Jam Recordings",
        "members": [],
        "wikidata": "Q36153",
        "albums": [("Good Girl Gone Bad", "2007"), ("Rated R", "2009"), ("Anti", "2016")]
    },
    {
        "name": "Whitney Houston",
        "type": "Person",
        "country": "US",
        "begin": "1983",
        "end": "2012",
        "genres": ["pop", "soul", "gospel", "R&B"],
        "label": "Arista Records",
        "members": [],
        "wikidata": "Q37079",
        "albums": [("Whitney Houston", "1985"), ("My Love Is Your Love", "1998"), ("I Look to You", "2009")]
    },
    {
        "name": "Massive Attack",
        "type": "Group",
        "country": "GB",
        "begin": "1988",
        "end": None,
        "genres": ["trip hop", "electronic", "downtempo"],
        "label": "Virgin Records",
        "members": ["Robert Del Naja", "Grant Marshall"],
        "wikidata": "Q183504",
        "albums": [("Blue Lines", "1991"), ("Mezzanine", "1998"), ("Heligoland", "2010")]
    },
    {
        "name": "Portishead",
        "type": "Group",
        "country": "GB",
        "begin": "1991",
        "end": None,
        "genres": ["trip hop", "electronic", "downtempo"],
        "label": "Go! Discs",
        "members": ["Beth Gibbons", "Geoff Barrow", "Adrian Utley"],
        "wikidata": "Q217294",
        "albums": [("Dummy", "1994"), ("Portishead", "1997"), ("Third", "2008")]
    },
    {
        "name": "Gorillaz",
        "type": "Group",
        "country": "GB",
        "begin": "1998",
        "end": None,
        "genres": ["alternative rock", "electronic", "hip hop"],
        "label": "Parlophone",
        "members": ["Damon Albarn", "Jamie Hewlett"],
        "wikidata": "Q259732",
        "albums": [("Gorillaz", "2001"), ("Demon Days", "2005"), ("Plastic Beach", "2010")]
    },
    {
        "name": "Joy Division",
        "type": "Group",
        "country": "GB",
        "begin": "1976",
        "end": "1980",
        "genres": ["post-punk", "gothic rock", "new wave"],
        "label": "Factory Records",
        "members": ["Ian Curtis", "Bernard Sumner", "Peter Hook", "Stephen Morris"],
        "wikidata": "Q183412",
        "albums": [("Unknown Pleasures", "1979"), ("Closer", "1980")]
    },
    {
        "name": "Aphex Twin",
        "type": "Person",
        "country": "GB",
        "begin": "1985",
        "end": None,
        "genres": ["electronic", "IDM", "ambient techno"],
        "label": "Warp Records",
        "members": [],
        "wikidata": "Q208638",
        "albums": [("Selected Ambient Works 85-92", "1992"), ("Richard D James Album", "1996"), ("Drukqs", "2001")]
    },
    {
        "name": "The Strokes",
        "type": "Group",
        "country": "US",
        "begin": "1998",
        "end": None,
        "genres": ["indie rock", "post-punk revival", "garage rock"],
        "label": "RCA Records",
        "members": ["Julian Casablancas", "Nick Valensi", "Albert Hammond Jr", "Nikolai Fraiture", "Fabrizio Moretti"],
        "wikidata": "Q183048",
        "albums": [("Is This It", "2001"), ("Room on Fire", "2003"), ("First Impressions of Earth", "2006")]
    },
    {
        "name": "LCD Soundsystem",
        "type": "Group",
        "country": "US",
        "begin": "2002",
        "end": None,
        "genres": ["dance-punk", "electronic rock", "post-punk revival"],
        "label": "DFA Records",
        "members": ["James Murphy"],
        "wikidata": "Q1352505",
        "albums": [("LCD Soundsystem", "2005"), ("Sound of Silver", "2007"), ("American Dream", "2017")]
    },
    {
        "name": "Bjork",
        "type": "Person",
        "country": "IS",
        "begin": "1986",
        "end": None,
        "genres": ["art pop", "electronic", "alternative", "experimental"],
        "label": "One Little Indian",
        "members": [],
        "wikidata": "Q47159",
        "albums": [("Debut", "1993"), ("Post", "1995"), ("Homogenic", "1997")]
    },
    {
        "name": "Tupac Shakur",
        "type": "Person",
        "country": "US",
        "begin": "1987",
        "end": "1996",
        "genres": ["hip hop", "gangsta rap", "west coast hip hop"],
        "label": "Death Row Records",
        "members": [],
        "wikidata": "Q155818",
        "albums": [("2Pacalypse Now", "1991"), ("All Eyez on Me", "1996"), ("Me Against the World", "1995")]
    },
]

print(f"Nombre d'artistes     : {len(ARTISTS_DATA)}")
print(f"Groupes               : {sum(1 for a in ARTISTS_DATA if a['type'] == 'Group')}")
print(f"Artistes solo         : {sum(1 for a in ARTISTS_DATA if a['type'] == 'Person')}")
print(f"Pays representes      : {len(set(a['country'] for a in ARTISTS_DATA))}")

artist_uris = {a['name']: a['wikidata'] for a in ARTISTS_DATA}

### Fonction utilitaire : créer une URI propre

In [ ]:
def safe_uri(name):
    clean = re.sub(r"[^\w\s\-]", "", name).strip()
    clean = re.sub(r"\s+", "_", clean)
    return EX[quote(clean, safe="_-")]

print(safe_uri("The Beatles"))
print(safe_uri("Kendrick Lamar"))
print(safe_uri("Jay-Z"))

### Création du graphe RDF

In [ ]:
g = Graph()
g.bind("ex",   EX)
g.bind("exo",  EXO)
g.bind("dbo",  DBO)
g.bind("wd",   WD)
g.bind("wdt",  WDT)
g.bind("owl",  OWL)
g.bind("rdfs", RDFS)
g.bind("foaf", FOAF)
g.bind("xsd",  XSD)

print(f"Triplets actuels : {len(g)}")

### Définition de l'ontologie

In [ ]:
classes = [
    (EXO.Artist,      "Artist",       None),
    (EXO.SoloArtist,  "SoloArtist",   EXO.Artist),
    (EXO.MusicGroup,  "MusicGroup",   EXO.Artist),
    (EXO.Album,       "Album",        None),
    (EXO.MusicGenre,  "MusicGenre",   None),
    (EXO.RecordLabel, "RecordLabel",  None),
    (EXO.Country,     "Country",      None),
]

for cls, label, parent in classes:
    g.add((cls, RDF.type,    OWL.Class))
    g.add((cls, RDFS.label,  Literal(label, lang="en")))
    if parent:
        g.add((cls, RDFS.subClassOf, parent))

proprietes = [
    (EXO.hasGenre,         "hasGenre",         EXO.Artist,     EXO.MusicGenre,  DBO.genre),
    (EXO.releasedAlbum,    "releasedAlbum",    EXO.Artist,     EXO.Album,       DBO.album),
    (EXO.signedTo,         "signedTo",         EXO.Artist,     EXO.RecordLabel, DBO.recordLabel),
    (EXO.originCountry,    "originCountry",    EXO.Artist,     EXO.Country,     DBO.birthPlace),
    (EXO.memberOf,         "memberOf",         EXO.SoloArtist, EXO.MusicGroup,  DBO.bandMember),
    (EXO.hasMember,        "hasMember",        EXO.MusicGroup, EXO.SoloArtist,  DBO.bandMember),
    (EXO.collaboratedWith, "collaboratedWith", EXO.Artist,     EXO.Artist,      None),
    (EXO.influencedBy,     "influencedBy",     EXO.Artist,     EXO.Artist,      DBO.influencedBy),
]

for uri, label, domaine, portee, equivalent in proprietes:
    g.add((uri, RDF.type,    OWL.ObjectProperty))
    g.add((uri, RDFS.label,  Literal(label, lang="en")))
    g.add((uri, RDFS.domain, domaine))
    g.add((uri, RDFS.range,  portee))
    if equivalent:
        g.add((uri, OWL.equivalentProperty, equivalent))

for uri, label, dtype in [
    (EXO.activeFrom,  "activeFrom",  XSD.gYear),
    (EXO.activeTo,    "activeTo",    XSD.gYear),
    (EXO.releaseYear, "releaseYear", XSD.gYear),
    (EXO.wikidataID,  "wikidataID",  XSD.string),
]:
    g.add((uri, RDF.type,   OWL.DatatypeProperty))
    g.add((uri, RDFS.label, Literal(label, lang="en")))

print(f"Triplets apres ontologie : {len(g)}")

### Fonction d'ajout d'un artiste

In [ ]:
def add_artist(graphe, info):
    name = info["name"]
    uri  = safe_uri(name)

    if info["type"] == "Group":
        graphe.add((uri, RDF.type, EXO.MusicGroup))
    else:
        graphe.add((uri, RDF.type, EXO.SoloArtist))
        graphe.add((uri, RDF.type, FOAF.Person))

    graphe.add((uri, RDFS.label, Literal(name, lang="en")))
    graphe.add((uri, FOAF.name,  Literal(name)))

    if info.get("wikidata"):
        graphe.add((uri, OWL.sameAs,     WD[info["wikidata"]]))
        graphe.add((uri, EXO.wikidataID, Literal(info["wikidata"])))

    if info.get("country"):
        pays_uri = EX[f"Country_{info['country']}"]
        graphe.add((pays_uri, RDF.type,   EXO.Country))
        graphe.add((pays_uri, RDFS.label, Literal(info["country"], lang="en")))
        graphe.add((uri, EXO.originCountry, pays_uri))

    if info.get("begin"):
        graphe.add((uri, EXO.activeFrom, Literal(info["begin"], datatype=XSD.gYear)))
    if info.get("end"):
        graphe.add((uri, EXO.activeTo,   Literal(info["end"],   datatype=XSD.gYear)))

    for genre_name in info.get("genres", []):
        genre_uri = EX[f"Genre_{quote(genre_name.replace(' ', '_'), safe='_')}"]
        graphe.add((genre_uri, RDF.type,   EXO.MusicGenre))
        graphe.add((genre_uri, RDFS.label, Literal(genre_name, lang="en")))
        graphe.add((uri, EXO.hasGenre, genre_uri))

    if info.get("label"):
        label_uri = safe_uri(f"Label_{info['label']}")
        graphe.add((label_uri, RDF.type,   EXO.RecordLabel))
        graphe.add((label_uri, RDFS.label, Literal(info["label"], lang="en")))
        graphe.add((uri, EXO.signedTo, label_uri))

    for titre, annee in info.get("albums", []):
        cle_album = re.sub(r"[^\w]", "_", titre)[:60]
        album_uri = EX[f"Album_{quote(cle_album, safe='_')}"]
        graphe.add((album_uri, RDF.type,        EXO.Album))
        graphe.add((album_uri, RDFS.label,      Literal(titre, lang="en")))
        graphe.add((album_uri, EXO.releaseYear, Literal(annee, datatype=XSD.gYear)))
        graphe.add((uri, EXO.releasedAlbum, album_uri))

    for membre_name in info.get("members", []):
        membre_uri = safe_uri(membre_name)
        graphe.add((membre_uri, RDF.type,   EXO.SoloArtist))
        graphe.add((membre_uri, RDF.type,   FOAF.Person))
        graphe.add((membre_uri, RDFS.label, Literal(membre_name, lang="en")))
        graphe.add((membre_uri, EXO.memberOf, uri))
        graphe.add((uri, EXO.hasMember, membre_uri))

    return uri

### Ajout des artistes dans le graphe

In [ ]:
print(f"Triplets avant : {len(g)}")

artist_uris = {}
for i, info in enumerate(ARTISTS_DATA, 1):
    uri = add_artist(g, info)
    artist_uris[info["name"]] = str(uri)

print(f"Triplets apres ajout des artistes : {len(g)}")

### Relations inter-artistes

In [ ]:
RELATIONS = [
    ("The Beatles",    "influencedBy",     "Elvis Presley",   "interviews 1964"),
    ("Nirvana",        "influencedBy",     "The Beatles",     "Kurt Cobain interviews 1993"),
    ("Kendrick Lamar", "influencedBy",     "Tupac Shakur",    "biographie documentee"),
    ("Radiohead",      "influencedBy",     "Pink Floyd",      "interviews Thom Yorke"),
    ("Gorillaz",       "influencedBy",     "The Beatles",     "Damon Albarn interviews"),
    ("Amy Winehouse",  "influencedBy",     "Nina Simone",     "interviews 2006"),
    ("Jay-Z",          "collaboratedWith", "Kanye West",      "Watch the Throne 2011"),
    ("Drake",          "collaboratedWith", "Kanye West",      "Find Your Love 2010"),
    ("Daft Punk",      "collaboratedWith", "Gorillaz",        "multiple collaborations"),
    ("Portishead",     "influencedBy",     "Massive Attack",  "Bristol scene"),
]

for artiste1, relation, artiste2, note in RELATIONS:
    uri1 = safe_uri(artiste1)
    uri2 = safe_uri(artiste2)
    if not any(True for _ in g.triples((uri2, RDF.type, None))):
        g.add((uri2, RDF.type,   EXO.Artist))
        g.add((uri2, RDFS.label, Literal(artiste2, lang="en")))
    g.add((uri1, EXO[relation], uri2))

print(f"Total triplets apres relations : {len(g)}")

### Statistiques finales

La KB contient 45 artistes répartis en 10 francophones et 35 anglophones.
Ce choix garantit une documentation riche sur Wikidata pour l'expansion SPARQL.

In [ ]:
total_triplets    = len(g)
entites_uniques   = len(set(g.subjects()))
predicats_uniques = len(set(g.predicates()))
groupes           = len(list(g.subjects(RDF.type, EXO.MusicGroup)))
artistes_solo     = len(list(g.subjects(RDF.type, EXO.SoloArtist)))
albums            = len(list(g.subjects(RDF.type, EXO.Album)))
genres            = len(list(g.subjects(RDF.type, EXO.MusicGenre)))

print("STATISTIQUES DE LA KB")
print(f"Triplets totaux    : {total_triplets}")
print(f"Entites uniques    : {entites_uniques}")
print(f"Predicats uniques  : {predicats_uniques}")
print(f"Groupes de musique : {groupes}")
print(f"Artistes solo      : {artistes_solo}")
print(f"Albums             : {albums}")
print(f"Genres             : {genres}")
print()
print("Verification des objectifs :")
print(f"  Triplets >= 100  : {total_triplets >= 100}   (valeur : {total_triplets})")
print(f"  Entites >= 50    : {entites_uniques >= 50}   (valeur : {entites_uniques})")

### Sauvegarde

- **Turtle (.ttl)** : format lisible par un humain
- **N-Triples (.nt)** : format brut pour l'entrainement KGE

In [ ]:
CHEMIN = "C:/Users/sandy/OneDrive/Desktop/Cours A4/Web Datamining/TD4_Project/"

g.serialize(CHEMIN + "private_kb.ttl", format="turtle")
print("Fichier sauvegarde : private_kb.ttl")

g.serialize(CHEMIN + "private_kb.nt", format="ntriples")
print("Fichier sauvegarde : private_kb.nt")

with open(CHEMIN + "artist_mapping.json", "w", encoding="utf-8") as f:
    json.dump(artist_uris, f, indent=2, ensure_ascii=False)
print("Fichier sauvegarde : artist_mapping.json")

print()
print("Apercu du fichier private_kb.ttl (20 premieres lignes) :")
with open(CHEMIN + "private_kb.ttl", "r", encoding="utf-8") as f:
    for i, ligne in enumerate(f):
        if i >= 20:
            break
        print(ligne, end="")